# RFM Customer Segmentation Analysis
**Portfolio Project — Vrushali Daptari**  
Tools: Python · Pandas · Matplotlib · Seaborn  
Dataset: Simulated E-commerce Retail Data (300 customers, 1,600+ transactions, Jan–Dec 2023)

---
## Project Overview
RFM (Recency, Frequency, Monetary) analysis is a proven marketing technique used to segment customers based on their purchasing behaviour.  
This project segments 300 customers into 4 lifecycle groups and maps targeted CRM/email strategies for each.

| Metric | Definition |
|--------|-----------|
| **Recency (R)** | Days since last purchase |
| **Frequency (F)** | Number of unique orders |
| **Monetary (M)** | Total revenue generated |


## Step 1: Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('ecommerce_retail_data.csv', parse_dates=['InvoiceDate'])

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()}")
print(f"Unique customers: {df['CustomerID'].nunique()}")
df.head()

## Step 2: Data Cleaning
Remove null CustomerIDs, cancelled orders, and invalid quantities/prices.

In [ ]:
# Check for nulls
print("Nulls before cleaning:")
print(df.isnull().sum())

# Remove nulls in CustomerID
df = df.dropna(subset=['CustomerID'])

# Remove negative quantities and prices (returns / errors)
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

# Create revenue column
df['TotalRevenue'] = (df['Quantity'] * df['UnitPrice']).round(2)

print(f"\nClean dataset: {len(df)} transactions")
df.describe()

## Step 3: Calculate RFM Metrics
Using snapshot date = day after last transaction.

In [ ]:
# Snapshot date: one day after last purchase
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Snapshot date: {snapshot_date.date()}")

# Aggregate per customer
rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo', 'nunique'),
    Monetary  = ('TotalRevenue', 'sum')
).reset_index()

rfm['Monetary'] = rfm['Monetary'].round(2)

print(f"\nRFM table shape: {rfm.shape}")
rfm.head(10)

## Step 4: Score Customers 1–5 (Quintiles)
- **Recency**: Lower days = higher score (5 = most recent)
- **Frequency & Monetary**: Higher = higher score (5 = best)

In [ ]:
# Score each metric 1-5
rfm['R_Score'] = pd.qcut(rfm['Recency'],   q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'),  q=5, labels=[1,2,3,4,5]).astype(int)

# Total RFM score (3–15)
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

print("Score distribution:")
print(rfm['RFM_Score'].value_counts().sort_index())
rfm[['CustomerID','Recency','Frequency','Monetary','R_Score','F_Score','M_Score','RFM_Score']].head(10)

## Step 5: Assign Customer Segments

| Segment | Score Range | Description |
|---------|------------|-------------|
| Champions | 13–15 | Best customers — recent, frequent, high spend |
| Loyal Customers | 10–12 | Regular buyers, good lifetime value |
| At Risk | 7–9 | Used to buy well, now fading — need win-back |
| Lost | 3–6 | No recent activity, low engagement |


In [ ]:
def assign_segment(score):
    if score >= 13: return 'Champions'
    elif score >= 10: return 'Loyal Customers'
    elif score >= 7:  return 'At Risk'
    else:             return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(assign_segment)

# Segment summary
seg_summary = rfm.groupby('Segment').agg(
    Customers      = ('CustomerID', 'count'),
    Avg_Recency    = ('Recency',    'mean'),
    Avg_Frequency  = ('Frequency',  'mean'),
    Avg_Monetary   = ('Monetary',   'mean'),
    Total_Revenue  = ('Monetary',   'sum')
).round(1).reset_index()

seg_summary['Revenue_%'] = (seg_summary['Total_Revenue'] / seg_summary['Total_Revenue'].sum() * 100).round(1)
seg_summary

## Step 6: Visualisations

In [ ]:
COLORS = {
    'Champions':       '#1D9E75',
    'Loyal Customers': '#7F77DD',
    'At Risk':         '#EF9F27',
    'Lost':            '#D85A30'
}
seg_order = ['Champions', 'Loyal Customers', 'At Risk', 'Lost']
seg_summary_ordered = seg_summary.set_index('Segment').reindex(seg_order).reset_index()

fig = plt.figure(figsize=(16, 12), facecolor='white')
fig.suptitle('RFM Customer Segmentation Analysis', fontsize=20, fontweight='bold', y=0.98)
gs = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# Chart 1: Customers per segment
ax1 = fig.add_subplot(gs[0, :2])
bars = ax1.bar(seg_summary_ordered['Segment'], seg_summary_ordered['Customers'],
               color=[COLORS[s] for s in seg_summary_ordered['Segment']], width=0.5, edgecolor='none')
for bar, val in zip(bars, seg_summary_ordered['Customers']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(int(val)), ha='center', fontsize=11, fontweight='bold', color='#333')
ax1.set_title('Customers per Segment', fontsize=13, fontweight='bold', pad=8)
ax1.set_ylabel('Count')
ax1.set_ylim(0, seg_summary_ordered['Customers'].max() * 1.15)
ax1.spines[['top','right']].set_visible(False)

# Chart 2: Revenue donut
ax2 = fig.add_subplot(gs[0, 2])
wedges, texts, autotexts = ax2.pie(
    seg_summary_ordered['Revenue_%'],
    labels=seg_summary_ordered['Segment'],
    colors=[COLORS[s] for s in seg_summary_ordered['Segment']],
    autopct='%1.0f%%', startangle=90,
    wedgeprops={'width': 0.6, 'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 8})
for at in autotexts:
    at.set_fontsize(9); at.set_fontweight('bold'); at.set_color('white')
ax2.set_title('Revenue Share\nby Segment', fontsize=13, fontweight='bold', pad=8)

# Chart 3: Scatter Recency vs Monetary
ax3 = fig.add_subplot(gs[1, :2])
for seg in seg_order:
    mask = rfm['Segment'] == seg
    ax3.scatter(rfm[mask]['Recency'], rfm[mask]['Monetary'],
                c=COLORS[seg], alpha=0.65, s=45, label=seg, edgecolors='none')
ax3.set_title('Recency vs Monetary Value', fontsize=13, fontweight='bold', pad=8)
ax3.set_xlabel('Recency (days)'); ax3.set_ylabel('Monetary Value (€)')
ax3.legend(fontsize=9, framealpha=0.3)
ax3.spines[['top','right']].set_visible(False)

# Chart 4: Avg monetary per segment
ax4 = fig.add_subplot(gs[1, 2])
h_bars = ax4.barh(seg_summary_ordered['Segment'], seg_summary_ordered['Avg_Monetary'],
                  color=[COLORS[s] for s in seg_summary_ordered['Segment']], height=0.5, edgecolor='none')
for bar, val in zip(h_bars, seg_summary_ordered['Avg_Monetary']):
    ax4.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
             f'€{val:.0f}', va='center', fontsize=9, fontweight='bold')
ax4.set_title('Avg Spend / Customer', fontsize=13, fontweight='bold', pad=8)
ax4.spines[['top','right']].set_visible(False)
ax4.set_xlim(0, seg_summary_ordered['Avg_Monetary'].max() * 1.2)

# Chart 5: RFM score distribution
ax5 = fig.add_subplot(gs[2, :])
scores = list(range(3, 16))
rfm_hist = rfm.groupby(['RFM_Score', 'Segment']).size().unstack(fill_value=0)
bottom = np.zeros(13)
for seg in seg_order:
    if seg in rfm_hist.columns:
        vals = [rfm_hist[seg].get(s, 0) for s in scores]
        ax5.bar(scores, vals, bottom=bottom, color=COLORS[seg], label=seg, edgecolor='white', linewidth=0.5)
        bottom += np.array(vals)
ax5.set_title('Customer Distribution by RFM Score', fontsize=13, fontweight='bold', pad=8)
ax5.set_xlabel('RFM Score (3=lowest, 15=highest)'); ax5.set_ylabel('Customers')
ax5.set_xticks(scores); ax5.legend(fontsize=9, framealpha=0.3)
ax5.spines[['top','right']].set_visible(False)

plt.savefig('rfm_charts.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print("Charts saved as rfm_charts.png")

## Step 7: Marketing Recommendations per Segment

This is the key differentiator — translating data into actionable CRM strategy.


In [ ]:
strategy = {
    'Segment': ['Champions', 'Loyal Customers', 'At Risk', 'Lost'],
    'Goal': [
        'Retain & maximise value',
        'Increase purchase frequency',
        'Win back before full churn',
        'Final reactivation attempt'
    ],
    'Channel': [
        'Email + SMS + Referral',
        'Email personalised',
        'Email win-back sequence',
        'Single email blast'
    ],
    'Action': [
        'VIP rewards, referral programme, early product access, upsell premium lines',
        'Personalised reorder reminders, loyalty points, exclusive flash sales',
        'Win-back email series (3 emails): discount offer + urgency + final CTA',
        'One final email: "We miss you" + strong offer. Unsubscribe if no open in 30 days.'
    ],
    'KPI to Track': [
        'LTV, repeat purchase rate, referral rate',
        'Purchase frequency, avg order value',
        'Email open rate, re-purchase rate',
        'Email open rate, unsubscribe rate'
    ]
}

strategy_df = pd.DataFrame(strategy)
pd.set_option('display.max_colwidth', 80)
strategy_df

## Step 8: Export Results

In [ ]:
# Export full RFM table to CSV for Looker Studio
rfm.to_csv('rfm_results.csv', index=False)
print(f"Exported rfm_results.csv — {len(rfm)} rows")
print("\nColumn reference for Looker Studio:")
print(rfm.columns.tolist())
print("\nFinal segment counts:")
print(rfm['Segment'].value_counts())

## Key Findings

| Segment | Customers | Revenue Share | Insight |
|---------|----------|--------------|---------|
| **Champions** | 72 (24%) | **65.7%** | Top 24% of customers generate 65%+ of revenue |
| **Loyal Customers** | 52 (17%) | 17.4% | High potential to convert into Champions |
| **At Risk** | 89 (30%) | 11.8% | Largest group — win-back campaign opportunity |
| **Lost** | 87 (29%) | 5.2% | Minimal revenue — final reactivation or sunset |

**Business Recommendation:**  
Focus 80% of CRM/email budget on Champions + Loyal Customers (combined 83.1% of revenue).  
Launch a targeted 3-email win-back sequence for At Risk customers to recover an estimated 8–12% incremental revenue.  
Sunset Lost customers after one final reactivation email to clean the list and improve deliverability.

---
*Project by Vrushali Daptari | Portfolio: vrushalidaptariportfolio.netlify.app*
